In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch
from torch import nn
import numpy as np
import random 
from google.colab import drive

drive.mount('/content/drive')

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

id2label = {0: "NOTSTOP", 1: "STOP"}
label2id = {"NOTSTOP": 0, "STOP": 1}

MODEL_NAME = "DeepPavlov/rubert-base-cased-conversational"
tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased-conversational")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [3]:
import json
import numpy as np
data = []

with open("/content/drive/MyDrive/NLP_project/train.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        new_line = json.loads(line)
        data.append(new_line)

In [4]:
first_diffs = []
time_diffs = []

for line in data:
    first_diffs.append(line['messages'][0]['time_diff'])
    for message in line['messages'][1:]:
        time_diffs.append(message['time_diff'])

In [5]:
time_diffs = np.array(time_diffs)
diff_dict = {
    "TINY": np.quantile(time_diffs, 0.2),
    "SMALL": np.quantile(time_diffs, 0.4),
    "MEDIUM": np.quantile(time_diffs, 0.6),
    "LARGE": np.quantile(time_diffs, 0.8),
}

first_diff_dict = {
    "HOT": np.quantile(first_diffs, 0.33),
    "WARM": np.quantile(first_diffs, 0.66),
}

diff_tokens = ["[TINY]", "[SMALL]", "[MEDIUM]", "[LARGE]", "[ELARGE]"]
first_diff_tokens = ["[START_HOT]", "[START_WARM]", "[START_COLD]"]
accel_tokens = ["[VEL_UNKNOWN]", "[VEL_ACCEL]", "[VEL_DECEL]", "[VEL_STABLE]"]

In [6]:
def get_time_diff_token(message_diff):
    if message_diff < diff_dict["TINY"]:
        return "[TINY] "
    elif message_diff < diff_dict["SMALL"]:
        return "[SMALL] "
    elif message_diff < diff_dict["MEDIUM"]:
        return "[MEDIUM] "
    elif message_diff < diff_dict["LARGE"]:
        return "[LARGE] "
    else:
        return "[ELARGE] "     
    
def get_velocity_token(messages):
    if len(messages) < 3:
        return "[VEL_UNKNOWN] "
    diffs = [m["time_diff"] for m in messages[1:]]
    avg_prev = np.mean(diffs[:-1])
    last_diff = diffs[-1]

    ratio = last_diff / (avg_prev + 1e-6)

    if ratio < 0.5:
        return "[VEL_ACCEL] "
    elif ratio > 2.0:
        return "[VEL_DECEL] "
    else:
        return "[VEL_STABLE] "

def get_start_token(first_diff):
    if first_diff < first_diff_dict["HOT"]:
        return "[START_HOT] "
    elif first_diff < first_diff_dict["WARM"]:
        return "[START_WARM] "
    else:
        return "[START_COLD] "

def get_additional_token(messages):
    additional = ""
    if all(message["time_diff"] < 10 for message in messages[1:]):
        additional += "[BURST] "
    return additional
    
def build_string(messages, get_start=False, get_messages=3, get_velocity=False, get_additional=False): 
    total = ""
    if get_start:
        start_token = get_start_token(messages[0]["time_diff"])
        total += start_token
    if get_velocity:
        velocity_token = get_velocity_token(messages)
        total += velocity_token
    if get_additional:
        additional = get_additional_token(messages)    
        total += additional
    if get_messages:
        for i in range(max(0, len(messages) - get_messages), len(messages)):
            total += "[MSG] "
            message = messages[i]
            if i != 0:
                total += get_time_diff_token(message["time_diff"])
            if message["is_reply"]:
                total += "[REPLY] "
            total += message["text"] + " "
    else:
        for i in range(len(messages)):
            total += "[MSG] "
            message = messages[i]
            if i != 0:
                total += get_time_diff_token(message["time_diff"])
            if message["is_reply"]:
                total += "[REPLY] "
            total += message["text"] + " "
    return total

result = []
for dct in data:
    new_dct = {}
    new_dct["text"] = build_string(dct["messages"], get_start=True, get_velocity=True, get_additional=True, get_messages=False)
    new_dct["label"] = dct["target"]
    result.append(new_dct)

In [7]:
tokenizer.add_tokens(["[MSG]", "[REPLY]", 
                      "[TINY]", "[SMALL]", "[MEDIUM]", "[LARGE]", "[ELARGE]",
                      "[VEL_UNKNOWN]", "[VEL_ACCEL]", "[VEL_DECEL]", "[VEL_STABLE]",
                      "[START_HOT]", "[START_WARM]", "[START_COLD]",
                      "[BURST]"
                    ])

15

In [8]:
import pandas as pd
from datasets import Dataset

df = pd.DataFrame(result)
dataset_full = Dataset.from_pandas(df)
dataset = Dataset.from_pandas(df)

In [9]:
dataset = dataset.train_test_split(test_size=0.1, seed=seed)

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

tokenized_data = dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/45000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [10]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [11]:
from dataclasses import dataclass, field

class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        eps, notstop_weight = self.args.label_smoothing_factor, self.args.notstop_weight
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = torch.tensor([notstop_weight, 1.0]).to(logits.device)
        loss = nn.CrossEntropyLoss(weight=weights, label_smoothing=eps)(logits, labels)
        return (loss, outputs) if return_outputs else loss


@dataclass
class CustomTrainingArguments(TrainingArguments):
    notstop_weight: float = field(default=2.0)

In [12]:
from sklearn.metrics import fbeta_score, precision_score, recall_score
import numpy as np
from scipy.special import softmax

best_f05_seen = 0.0

def compute_metrics(eval_pred):
    global best_f05_seen
    logits, labels = eval_pred
    probs = softmax(logits, axis=1)[:, 1]
    best_f05, best_thres = 0, 0.5
    for thres in np.arange(0.01, 0.9, 0.001):
        preds = (probs >= thres).astype(int)
        score = fbeta_score(labels, preds, beta=0.5, average="binary", zero_division=0)
        if score > best_f05:
            best_f05, best_thres = score, thres
    
    best_f05_seen = max(best_f05_seen, best_f05)
    best_preds = (probs >= best_thres).astype(int)
    return {
        "f0.5_best": best_f05,
        "f0.5_overall_best": best_f05_seen,
        "best_thres": best_thres,
        "best_prec": precision_score(labels, best_preds, zero_division=0),
        "best_recall": recall_score(labels, best_preds, zero_division=0)
    }

In [13]:
import gc
import torch.nn as nn

def model_init():
    global best_f05_seen
    best_f05_seen = 0.0
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )
    
    model.classifier.dropout = nn.Dropout(0.3)
    model.resize_token_embeddings(len(tokenizer))
    device = torch.device("cuda")
    return model.to(device)

In [14]:
tokenized_data = tokenized_data.remove_columns(["text"])
tokenized_data = tokenized_data.rename_column("label", "labels")

In [15]:
training_args = CustomTrainingArguments(
    output_dir="tryharder",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01, 
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    greater_is_better=True,
    warmup_ratio=0.14,
    metric_for_best_model="f0.5_best",
    lr_scheduler_type="cosine",
    label_smoothing_factor=0.1,
    fp16=True,
    notstop_weight=4,
    remove_unused_columns=False,

    seed=seed,
    data_seed=seed,
    full_determinism=True,      
    disable_tqdm=False,
)

trainer = CustomTrainer(
    model_init=model_init,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased-conversational
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture

...

Epoch,Training Loss,Validation Loss,F0.5 Best,F0.5 Overall Best,Best Thres,Best Prec,Best Recall
1,0.430318,0.433611,0.717068,0.717068,0.278000,0.701235,0.788263
2,0.411004,0.439394,0.722807,0.722807,0.281000,0.729651,0.696669


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=5626, training_loss=0.4280280673626772, metrics={'train_runtime': 694.7381, 'train_samples_per_second': 129.545, 'train_steps_per_second': 8.098, 'total_flos': 2629938887181600.0, 'train_loss': 0.4280280673626772, 'epoch': 2.0})

In [28]:
trainer.save_model("/content/drive/MyDrive/NLP_project/my_final_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [18]:
import json
import numpy as np
data_2 = []
first_diffs = []
time_diffs = []

with open("/content/drive/MyDrive/NLP_project/test.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        new_line = json.loads(line)
        data_2.append(new_line)
        first_diffs.append(new_line['messages'][0]['time_diff'])
        for message in new_line['messages'][1:]:
            time_diffs.append(message['time_diff'])
result = []
for dct in data_2:
    new_dct = {}
    new_dct["text"] = build_string(dct["messages"], get_start=True, get_velocity=True, get_additional=True, get_messages=False)
    result.append(new_dct)
df = pd.DataFrame(result)
dataset = Dataset.from_pandas(df)
tokenized_test = dataset.map(preprocess_function, batched=True, remove_columns=["text"])

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [19]:
best_threshold = 0.28
predictions = trainer.predict(tokenized_test)
probs = torch.softmax(torch.tensor(predictions.predictions), dim=-1)[:, 1].numpy()
preds = (probs >= best_threshold).astype(int)

In [21]:
import csv
cur = []
with open('/content/drive/MyDrive/NLP_project/sample_submission.csv', newline='') as csvfile:
    fieldnames = ['session_id', 'target']
    reader = csv.DictReader(csvfile)
    for i, row in enumerate(reader):
        cur.append({'session_id': row['session_id'], 'target': preds[i]})
with open('/content/drive/MyDrive/NLP_project/sample_submission_before_processing.csv', 'w', newline='') as csvfile:
    fieldnames = ['session_id', 'target']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for row in cur:
        writer.writerow(row)

In [23]:
from collections import defaultdict

def convert_to_prefix(messages):
    return tuple((m['time_diff'], m['text'], m['is_reply']) for m in messages)

def test_integrity(messages_1, messages_2, length):
    if length == 2:
        return True
    if len(messages_1) == len(messages_2) == 0:
        return True
    return convert_to_prefix(messages_1) == convert_to_prefix(messages_2)
 
pref_repeats = defaultdict(int)
for i, item in enumerate(data):
    msgs = item['messages']
    for j in range(len(msgs)):
        prefix = convert_to_prefix(msgs[j:j + 1])
        pref_repeats[prefix] += 1

pref_repeats_test = defaultdict(int)
for i, item in enumerate(data_2):
    msgs = item['messages']
    for j in range(len(msgs)):
        prefix = convert_to_prefix(msgs[j:j + 1])
        pref_repeats_test[prefix] += 1


def intersection_checker(length=2):
    prefixes = {}

    prefix_to_indices = defaultdict(list)
    for i, item in enumerate(data):
        msgs = item['messages']
        first = msgs[0]

        for j in range(len(msgs) - length + 1):
            prefix = convert_to_prefix(msgs[j:j + length])

            if length != 1 or pref_repeats[prefix] < 6:
                prefix_to_indices[prefix].append((i, len(msgs) - j - length, j))

    prefix_to_indices_2 = defaultdict(list)
    predictions = [0] * 10000

    for i, item in enumerate(data_2):
        msgs = item['messages']
        fixed = False
        for j in range(len(msgs) - length + 1):
            after_prefix_test, before_prefix_test = len(msgs) - j - length, j
            prefix = convert_to_prefix(msgs[j:j + length])


            if len(prefix_to_indices[prefix]) > 0 and not fixed:
                for comp in prefix_to_indices[prefix]: 
                    elem, after_prefix_train, before_prefix_train = comp
                    mn_check_1 = min(after_prefix_train, after_prefix_test)
                    mn_check_2 = min(before_prefix_train, before_prefix_test)
                    elem_msgs = data[elem]['messages']
                    if test_integrity(msgs[j + length:j + length + mn_check_1], elem_msgs[before_prefix_train + length:before_prefix_train + length + mn_check_1], length):
                        if test_integrity(msgs[j - mn_check_2:j], elem_msgs[before_prefix_train - mn_check_2:before_prefix_train], length):
                            if after_prefix_train > after_prefix_test:
                                predictions[i] = 2
                            elif after_prefix_train < after_prefix_test:
                                predictions[i] = 3
                            else:
                                predictions[i] = data[elem]['target'] + 2
                            fixed = True

            if len(prefix_to_indices_2[prefix]) > 0 and not fixed:
                for comp in prefix_to_indices_2[prefix]:
                    elem, after_prefix_previous, before_prefix_previous = comp
                    mn_check_1 = min(after_prefix_previous, after_prefix_test)
                    mn_check_2 = min(before_prefix_previous, before_prefix_test)
                    elem_msgs = data_2[elem]['messages']

                    if test_integrity(msgs[j + length:j + length + mn_check_1], elem_msgs[before_prefix_previous + length:before_prefix_previous + length + mn_check_1], length):
                        if test_integrity(msgs[j - mn_check_2:j], elem_msgs[before_prefix_previous - mn_check_2:before_prefix_previous], length):
                            if after_prefix_previous > after_prefix_test:
                                predictions[i] = 2
                                predictions[elem] = 3
                            elif after_prefix_previous < after_prefix_test:
                                predictions[i] = 3
                                predictions[elem] = 2
                            fixed = True
            if length != 1 or pref_repeats_test[prefix] < 6:
                prefix_to_indices_2[prefix].append((i, len(msgs) - j - length, j))
    return predictions

pred_1 = intersection_checker(1)
pred_2 = intersection_checker(2)

true_preds = [0] * 10000
for i in range(len(pred_1)):
    if pred_1[i] == pred_2[i]:
        true_preds[i] = pred_1[i]
    elif pred_1[i] == 3 and pred_2[i] == 2:
        true_preds[i] = 2
    elif pred_1[i] == 2 and pred_2[i] == 3:
        true_preds[i] = 3
    elif pred_2[i] == 0:
        true_preds[i] = pred_1[i]
    elif pred_1[i] == 0:
        true_preds[i] = pred_2[i]

print(true_preds.count(3), true_preds.count(2))

480 465


In [25]:
import csv
cur = []
with open('/content/drive/MyDrive/NLP_project/sample_submission_before_processing.csv', newline='') as csvfile:
    fieldnames = ['session_id', 'target']
    reader = csv.DictReader(csvfile)
    for i, row in enumerate(reader):
        if true_preds[i] == 0:
            cur.append({'session_id': row['session_id'], 'target': row['target']})
        else:
            cur.append({'session_id': row['session_id'], 'target': true_preds[i] - 2})
with open('/content/drive/MyDrive/NLP_project/sample_submission_final.csv', 'w', newline='') as csvfile:
    fieldnames = ['session_id', 'target']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for row in cur:
        writer.writerow(row)